In [1]:
import os 
from langchain_core.prompts import ChatMessagePromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import OllamaLLM
from langchain_core.runnables import RunnablePassthrough,RunnableBranch,Runnable,RunnableParallel
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_core.tools import tool
import re
from typing import List,Dict,Union

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [3]:
@tool
def add_numbers(inputs:str) ->dict:
    """
    Adds a list of number provided in the input string.
    Parameters:
    -inputs (str):
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add number 10,20, and 30."
    Example Output:
    {"result":60}
    """
    # Use regular expression to extract all number from input
    numbers=[int(num) for num in re.findall(r'\d+',inputs)]
    result=sum(numbers)
    return {'result':result}




#### The above functions will now act as a tool. You can inspect the tool's schema and other properties using

In [4]:
print(f'{"="*30}Name of the tool {"="*30}\n',add_numbers.name)
print(f'{"="*30}Description of the tool {"="*30}\n',add_numbers.description)
print(f'{"="*30} Args of the tool {"="*30}\n',add_numbers.args)

==============================Name of the tool ==============================
 add_numbers
==============================Description of the tool ==============================
 Adds a list of number provided in the input string.
Parameters:
-inputs (str):
string, it should contain numbers that can be extracted and summed.
Returns:
- dict: A dictionary with a single key "result" containing the sum of the numbers.
Example Input:
"Add number 10,20, and 30."
Example Output:
{"result":60}
============================== Args of the tool ==============================
 {'inputs': {'title': 'Inputs', 'type': 'string'}}


#### You can call the tool using the invoke method

In [5]:
test_input=' 1,29 and 32'
print(add_numbers.invoke(test_input))

{'result': 62}


In [6]:
from langchain_core.tools import Tool

In [7]:
import re
from langchain_core.tools import tool

@tool
def add_numbers(inputs) -> dict:
    """
    Add numbers from either a string or a list.
    """

    if isinstance(inputs, list):
        result = sum(float(x) for x in inputs)

    elif isinstance(inputs, str):
        numbers = [float(num) for num in re.findall(r'\d+(?:\.\d+)?', inputs)]
        result = sum(numbers)

    else:
        raise ValueError("Input must be a string or list.")

    return {"result": result}

In [8]:
add_tool = Tool(
    name='AddTool',
    func=add_numbers.func,
    description='Add a list of numbers and returns the results'
)

print('tool object', add_tool)


tool object name='AddTool' description='Add a list of numbers and returns the results' func=<function add_numbers at 0x0000017D932C74C0>


#### In this example the toll has two inputs : a string containing the number to add, and second boolean input that determines whether to sum the absolute values of those numbers.

In [9]:
@tool
def add_numbers_with_options(numbers:List[float],absolute:bool=False) ->float:
    """
    Adds a list of numbers provided as input.
    
    Parameters:
    -number (List[float]) : A list of number to be summed.
    =absolute (bool) : If True, use the absolute value of the number before summing.
    
    Returns:
    -float: The total sum of the numbers.
    """
    if absolute:
        numbers=[abs(num) for num in numbers]
    return sum(numbers)

In [10]:
print(add_numbers_with_options.invoke({'numbers':[1,2,-3],'absolute':False}))
print(add_numbers_with_options.invoke({'numbers':[1,2,-3],'absolute':True}))

0.0
6.0


#### Improved tool return types with Python typing

In [11]:
@tool
def sum_numbers_with_complex_output(inputs:str) ->Dict[str,Union[float,str]]:
    """
    Extract and sum all the integers and decimal numbers from the inputs strings.
    
    Parameters:
    -input (str) : A string that may contains numeric values.
    
    Returns:
    -dict : A dictionary with the key 'result'. if numbers are found, the value is ther sum(float)
            if no number are found or an error occurs, the value is a corresponding message (str)
    
    Example Input:
    'Add the numbers 10,20.5 and -30'
    
    Example Output:
    {'result':0.5}
    """
    matches=re.findall(r'-?\d+(?:\.\d+)?',inputs)
    if not matches:
        return {'result':'No number found in inputs'}
    try:
        numbers=[float(num) for num in matches]
        total=sum(numbers)
        return {'result':total}
    except Exception as e:
        return {'result':f'Error during summation:{str(e)}'}
        
        
    

In [12]:
print(sum_numbers_with_complex_output.invoke('what is the sum of 33, 32.5 and 30'))

{'result': 95.5}


### `initialize_agent`

When you set up an agent, you're connecting tools and an LLM to work together seamlessly. The agent uses the LLM to understand what needs to be done and decides which tool to use based on the task. Here's a simple overview of the key parts:


#### **Relationship between agent and LLM**
- The **agent** acts as the decision-maker, figuring out which tools to use and when.
- The **LLM** is the reasoning engine. It:
  - Interprets the user's input.
  - Helps the agent make decisions.
  - Generates a response based on the output of the tools.

Think of the agent as the manager assigning tasks and the LLM as the brain solving problems or delegating work.

---

#### **Key parameters of `initialize_agent`**

1. **`tools`**- see above 

2.  **`llm`** see above 

3. **`agent`**:
   - Specifies the reasoning framework for the agent.
   - `"zero-shot-react-description"` enables:
     - **Zero-shot reasoning**: The agent can solve tasks it hasn't seen before by thinking through the problem step by step.
     - **React framework**: A logical loop of:
       - **Reason** → Think about the task.
       - **Act** → Use a tool to perform an action.
       - **Observe** → Check the tool's output.
       - **Plan** → Decide what to do next.

4. **`verbose`**:
   - If `True`, it prints detailed logs of the agent’s thought process.
   - Useful for debugging or understanding how the agent makes decisions.




In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[add_tool],
    system_prompt='You are a helpful assistant',
    
)


response=agent.invoke(
    {'messages':[{'role':'user',"content":"In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total."}]}
)

print(response["messages"][-1].content)

Confirmed! The total GDP of the US, Canada, and Mexico combined in 2023 was approximately **$31.65 trillion**.


#### Orchestrating multiple tools with an agent: Mathematical toolkit

#### Subtract

In [14]:
@tool 
def subtract_numbers(inputs: str) ->dict:
    """
    Extract numbers from string, negates the first number, and successively subtract the remaining numbers in the list
    
    This function is designed to handle input in string format, where numbers are separated by spaces, commas or other delimiters. It parses the string, extracts valid numeric values and performs a step by step subtractions operations with the first number negated.
    
    Parameters:
    -inputs (str):
    A string containing number to subtract. The string may include spaces,comma,
    or other delimeters between the numbers.
    
    Returns:
    -dict:
    A dictionary containing the key 'result' with calculated results as values.
    if no valid number are found in the input string, the result default is 0.
    
    Example Input:
    '100,20,10'
    
    Example Output:
    {'result':70}
    """
    
    # Extract number from the string
    numbers = [int(num) for num in inputs.replace(",","").split() if num.isdigit()]
    
    # if no number found, return 0
    if not numbers:
        return {'result':0}
    #start with the first number negated
    result= -1*numbers[0]
    
    for num in numbers[1:]:
        result+=num
    return {'result':result}

In [15]:
print(subtract_numbers.invoke('subtract 100 from 40'))

{'result': -60}


#### Multiply

In [16]:
@tool 
def multiply_numbers(inputs: str) -> dict:
    """
    Extract the number from the string and calculates their products
    
    Parameters:
    -inputs(str) :A string containing numbers separated by comma,spaces and other delimiters
    
    Returns:
    -dict : A dictionary with the key 'result' and the product of the numbers as keys
    
    Example Input:
    "3,4,10"
    
    Example Output:
    {'result':120}
    
    Notes:
    - if no numbers are found, the result default is 1.
    """
    
    # Extract the numbers from the string
    numbers=[int(num) for num in inputs.replace(',','').split() if num.isdigit()]
    
    # if no number are found 
    if not numbers:
        return {'result':1}
    
    #calculates the product of the numbers:
    result=1
    for num in numbers:
        result*=num
    return {'result':result}

In [17]:
print(multiply_numbers.invoke('what is the product of 3, 5 , 4'))

{'result': 60}


#### Divide

In [18]:
@tool
def divide_numbers(input:str) -> dict:
    """
    Extract the numbers from a string and calculates the result of dividing the first number
    by the subsequent numbers in sequence.
    
    Parameters:
    -input(str): A string containing numbers separated by space, comma, and other delimiters
    
    Returns:
    -dict: a dictionary with the key as 'result' containing the quotient
    
    Example Input:
    "100 , 5, 2"
    
    Example Output:
    {'result':10}
    
    Notes:
    - if no number are found, the result defaults to 0.
    - Division by zero will raise a error
    """
    
    # Extract the numbers from the string
    numbers=[int(num) for num in input.replace(',','').split() if num.isdigit()]
    
    if not numbers:
        return {'result':0}
    
    result=numbers[0]
    for num in numbers[1:]:
        result/=num
    return {'result':result}
    

In [19]:
print(divide_numbers.invoke('100 is divided by 5 and 20'))

{'result': 1.0}


#### Building the agent

In [20]:
tools=[add_numbers,subtract_numbers,multiply_numbers,divide_numbers]

In [21]:
from langgraph.prebuilt import create_react_agent

In [22]:
# create the agent with all the tools

math_agent= create_react_agent(
    model =llm,
    tools=tools,
    prompt='You are helpful mathematical assistant that can perform various operationsuse your tools precisely and explain the reasoning clearly ' 
)

C:\Users\Pappu Yadav\AppData\Local\Temp\ipykernel_18976\3576574577.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent= create_react_agent(


In [23]:
response=math_agent.invoke({
    'messages':[('human','what is 24 divided by 4 and then 20 is added')]
})

#get the final answer
print(response['messages'][-1].content)

**The answer is 26.**

Here's the breakdown:
- 24 ÷ 4 = 6
- 6 + 20 = **26**


#### Exploring Langchain's built in tools

In [24]:
#using the wikipedia tool
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\Pappu Yadav\AppData\Local\Temp\ipykernel_18976\2343339862.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import WikipediaAPIWrapper


In [55]:
from langchain_core.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper


from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

@tool
def web_search(query: str) -> str:
    """Search the web for factual information."""
    return search.invoke(query)

In [57]:
result = web_search.invoke("what is the capital city of nepal")
print(result)


Kathmandu served as the royal capital of the Kingdom of Nepal and is home to numerous palaces, temples, and gardens reflecting its rich heritage. Since 1985, it has hosted the headquarters of the South Asian Association for Regional Cooperation (SAARC). 2 days ago · Where in the World is Kathmandu found? Kathmandu is the capital of Nepal (Federal Democratic Republic of Nepal), situated in the Southern Asia subregion of Asia. In Kathmandu, the currency used is Nepalese rupee (₨), which is the official currency used in Nepal. The Latitude, Longitude cordinates of Kathmandu are 27.7017, 85.3206. Kathmandu is the capital and eldest metropolitan city of Nepal, located in the Kathmandu Valley in the Himalayas. It is the urban core of the Central Region and the gateway to Nepal Tourism, with a rich history and culture of over 2000 years. Kathmandu is the capital of Nepal and the largest metropolis in the country. It has a rich and diverse cultural heritage, many UNESCO World Heritage Sites, a

In [58]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, subtract_numbers, multiply_numbers, divide_numbers, web_search]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

C:\Users\Pappu Yadav\AppData\Local\Temp\ipykernel_18976\3026667015.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  math_agent_updated = create_react_agent(


In [59]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")


Message sequence:

--- Message 1 ---
Type: HumanMessage
Content: What is the population of Canada? Multiply it by 0.75
Name: None

--- Message 2 ---
Type: AIMessage
Content: I'll first look up the population of Canada, then multiply it by 0.75.
Name: None
Tool calls: [{'name': 'web_search', 'args': {'query': 'population of Canada 2024'}, 'id': '7ce3bcad-fae2-4ce8-a01d-9b6d00fad21b', 'type': 'tool_call'}]

--- Message 3 ---
Type: ToolMessage
Content: 4 days ago - A population estimate for 2024 put the total number of people in Canada at 41,012,563. 1 day ago - Canada ranks 37th by population among countries of the world, comprising about 0.5% of the world's total, with about 41.5 million Canadians as of Q1 2026. Despite being the second-largest country by total area (fourth-largest by land area), ... December 4, 2025 - Okay, so the big question: What’s the current population of Canada? As of 2024, Canada’s population is estimated to be around 40 million people. Yeah, that’s a pretty bi